# Week 2

Original Code

solve.py

In [10]:
import pydot
from collections import deque
import argparse
import os

# Set path to Graphviz bin folder (customize as needed)
os.environ["PATH"] += os.pathsep + 'C:\\Program Files\\Graphviz\\bin'
# Possible moves (missionaries, cannibals)
options = [(1, 0), (0, 1), (1, 1), (0, 2), (2, 0)]

# Dictionary to keep track of parent for graph edges
Parent = dict()

# Initialize pydot graph for visualizing the state space tree
graph = pydot.Dot(
    graph_type='graph', strict=False, bgcolor="#fff3af",
    label="fig: Missionaries and Cannibal State Space Tree",
    fontcolor="red", fontsize="24", overlap="true"
)

# Node counter
i = 0

def is_valid_move(number_missionaries, number_cannibals):
    """Check if numbers are within 0–3 inclusive."""
    return 0 <= number_missionaries <= 3 and 0 <= number_cannibals <= 3

def is_start_state(number_missionaries, number_cannibals, side):
    return (number_missionaries, number_cannibals, side) == (3, 3, 1)

def is_goal_state(number_missionaries, number_cannibals, side):
    return (number_missionaries, number_cannibals, side) == (0, 0, 0)

def number_of_cannibals_exceeds(number_missionaries, number_cannibals):
    m_right = 3 - number_missionaries
    c_right = 3 - number_cannibals
    return (number_missionaries > 0 and number_cannibals > number_missionaries) or \
           (m_right > 0 and c_right > m_right)

def draw_edge(number_missionaries, number_cannibals, side, depth_level, node_num):
    current = (number_missionaries, number_cannibals, side, depth_level, node_num)
    parent = Parent.get(current)
    if parent is not None:
        u = pydot.Node(str(parent), label=str(parent[:3]))
        v = pydot.Node(str(current), label=str(current[:3]))
        graph.add_node(u); graph.add_node(v)
        graph.add_edge(pydot.Edge(str(parent), str(current), dir='forward'))
    else:
        v = pydot.Node(str(current), label=str(current[:3]))
        graph.add_node(v)
    return parent, current

def generate(max_depth=20):
    """BFS over state space, building the graph."""
    global i
    q = deque()
    node_num = 0
    q.append((3, 3, 1, 0, node_num))
    Parent[(3, 3, 1, 0, node_num)] = None

    while q:
        m, c, s, depth, num = q.popleft()
        draw_edge(m, c, s, depth, num)

        if is_goal_state(m, c, s):
            return True
        if depth >= max_depth:
            return False

        op = -1 if s == 1 else 1
        expanded = False

        for x, y in options:
            nm, nc, ns = m + op*x, c + op*y, 1 - s
            node = (nm, nc, ns, depth+1, i+1)
            if node not in Parent and is_valid_move(nm, nc):
                expanded = True
                i += 1
                q.append(node)
                Parent[node] = (m, c, s, depth, num)

        if not expanded:
            # dead end
            pass

    return False

def write_image(file_name="state_space"):
    try:
        graph.write_png(f"{file_name}.png")
        print(f"File {file_name}.png written.")
    except Exception as e:
        print("Error writing image:", e)


class Solution:
    def __init__(self):
        self.start = (3, 3, 1)
        self.goal = (0, 0, 0)
        self.options = options
        self.graph = pydot.Dot(
            graph_type='graph', bgcolor="#fff3af",
            label="Missionaries & Cannibals State Space", fontcolor="red", fontsize="24"
        )
        self.visited = {}

    def is_valid(self, m, c):
        return 0 <= m <= 3 and 0 <= c <= 3

    def exceeds(self, m, c):
        return number_of_cannibals_exceeds(m, c)

    def draw_node(self, m, c, s, depth):
        node = (m, c, s, depth)
        parent = Parent.get(node)
        if parent:
            u = pydot.Node(str(parent), label=str(parent[:3]))
            v = pydot.Node(str(node), label=str(node[:3]))
            self.graph.add_node(u); self.graph.add_node(v)
            self.graph.add_edge(pydot.Edge(str(parent), str(node), dir='forward'))
        else:
            v = pydot.Node(str(node), label=str(node[:3]))
            self.graph.add_node(v)
        return node

    def bfs(self):
        q = deque([self.start + (0,)])
        self.visited[self.start] = True
        while q:
            m, c, s, depth = q.popleft()
            node = self.draw_node(m, c, s, depth)

            if (m, c, s) == self.goal:
                return True
            if self.exceeds(m, c):
                continue

            op = -1 if s == 1 else 1
            for x, y in self.options:
                nm, nc, ns = m + op*x, c + op*y, 1 - s
                state = (nm, nc, ns)
                if state not in self.visited and self.is_valid(nm, nc):
                    self.visited[state] = True
                    Parent[(nm, nc, ns, depth+1)] = (m, c, s, depth)
                    q.append((nm, nc, ns, depth+1))
        return False

    def dfs(self, m, c, s, depth=0):
        self.visited[(m, c, s)] = True
        node = self.draw_node(m, c, s, depth)

        if (m, c, s) == self.goal:
            return True
        if self.exceeds(m, c):
            return False

        op = -1 if s == 1 else 1
        for x, y in self.options:
            nm, nc, ns = m + op*x, c + op*y, 1 - s
            state = (nm, nc, ns)
            if state not in self.visited and self.is_valid(nm, nc):
                Parent[(nm, nc, ns, depth+1)] = (m, c, s, depth)
                if self.dfs(nm, nc, ns, depth+1):
                    return True
        return False

    def show_solution(self):
        path = []
        cur = self.goal
        while cur:
            path.append(cur)
            # find parent mapping
            for k, v in Parent.items():
                if k[:3] == cur:
                    cur = v[:3] if v else None
                    break
            else:
                cur = None
        for step, state in enumerate(reversed(path)):
            print(f"Step {step}: {state}")

# Example usage:
# if generate():
#     write_image("state_space")
# sol = Solution()
# if sol.bfs():
#     sol.show_solution()
# sol.graph.write_png("bfs_solution.png")


main.py

In [11]:
from solve import Solution
import argparse
import itertools

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-m", "--method", required=False,
                        help="Specify which method to use")
    parser.add_argument("-l", "--legend", required=False,
                        help="Specify if you want to display legend on graph")
    args = vars(parser.parse_args())

    solve_method = args.get("method", "bfs")
    legend_flag  = args.get("legend", False)

    s = Solution()
    if s.solve(solve_method):
        # Display solution on console
        s.show_solution()

        # Compose output file name
        output_file_name = f"{solve_method}"
        if legend_flag:
            if legend_flag[0].upper() == 'T':
                output_file_name += "_legend.png"
                s.draw_legend()
            else:
                output_file_name += ".png"
        else:
            output_file_name += ".png"

        # Write state space tree image
        s.write_image(output_file_name)
    else:
        raise Exception("No solution found")

if __name__ == "__main__":
    main()


ImportError: cannot import name 'Solution' from 'solve' (c:\Users\khoi\OneDrive\Documents\Introduction-to-AI\.venv\Lib\site-packages\solve\__init__.py)